# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [3]:
import os

from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('OPENAI_API_KEY')})

In [4]:
import requests
from langchain_community.document_loaders import PyPDFLoader

file_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
pdf_path = "Managing_Oneself_Drucker_HBR.pdf"

response = requests.get(file_url)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

client = OpenAI(default_headers={"x-api-key": os.getenv('OPENAI_API_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

class SummaryOutput(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="Why this article is relevant for an AI professional")
    Summary: str = Field(description="A concise summary of the article (under 1000 tokens)")
    Tone: str = Field(description="The tone used to write the summary")
    InputTokens: int = Field(default=0, description="Number of input tokens")
    OutputTokens: int = Field(default=0, description="Number of output tokens")

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": (
                "You are a lawyer. Read the article and produce a structured summary. "
                "Return the author, title, relevance for AI professionals, a concise summary, and the tone used. "
                "Write the summary in Legalese. "
                "The Summary field must be no longer than 1000 tokens. "
                "Use only information grounded in the source article."
            ),
        },
        {
            "role": "user",
            "content": f"ARTICLE:\n{document_text}",
        },
    ],
    text_format=SummaryOutput,
)


summary_output = response.output_parsed
summary_output.InputTokens = response.usage.input_tokens
summary_output.OutputTokens = response.usage.output_tokens

print(summary_output)
print("InputTokens:", response.usage.input_tokens)
print("OutputTokens:", response.usage.output_tokens)

from IPython.display import display, Markdown

display(Markdown(f"""
### Title
{summary_output.Title}

### Author
{summary_output.Author}

### Relevance
{summary_output.Relevance}

### Summary
{summary_output.Summary}

### Tone
{summary_output.Tone}
"""))



Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is significant for AI professionals as it outlines the necessity of self-management and personal accountability in career progression, themes applicable to AI professionals navigating a rapidly evolving and ambiguous job landscape.' Summary='In the contemporary era characterized by unprecedented opportunities, the onus of career management falls squarely upon the individual. Knowledge workers are urged to adopt the role of their own Chief Executive Officer, thereby necessitating a profound understanding of personal strengths, weaknesses, values, performance modes, and ideal environments for efficacy. The document delineates methodologies such as feedback analysis to uncover these personal insights, emphasizing that successful navigation within the knowledge economy requires acknowledging one’s strengths and actively seeking environments aligned with one’s values and capabilities. Furthermore, individuals are enc


### Title
Managing Oneself

### Author
Peter F. Drucker

### Relevance
This article is significant for AI professionals as it outlines the necessity of self-management and personal accountability in career progression, themes applicable to AI professionals navigating a rapidly evolving and ambiguous job landscape.

### Summary
In the contemporary era characterized by unprecedented opportunities, the onus of career management falls squarely upon the individual. Knowledge workers are urged to adopt the role of their own Chief Executive Officer, thereby necessitating a profound understanding of personal strengths, weaknesses, values, performance modes, and ideal environments for efficacy. The document delineates methodologies such as feedback analysis to uncover these personal insights, emphasizing that successful navigation within the knowledge economy requires acknowledging one’s strengths and actively seeking environments aligned with one’s values and capabilities. Furthermore, individuals are encouraged to cultivate relationships within organizations, enhancing performance through understanding and adapting to the work styles of peers and superiors. The discourse advocates for strategic contributions and ongoing self-development, steering clear of assignments incongruent with personal competencies. This self-aware, proactive approach ensures sustained engagement and productivity throughout an extended career, potentially spanning five decades. Ultimately, the text posits that the ability to manage oneself effectively is not only pivotal for individual excellence but is increasingly necessary in an environment where organizational longevity cannot be assumed.

### Tone
Formal, analytical, and instructive.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [6]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('OPENAI_API_KEY')})
judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv("OPENAI_API_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_output.Summary
)


In [7]:
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    include_reason=True,
    assessment_questions=[
        "Does the summary capture the main argument of the article?",
        "Does the summary include the most important supporting ideas?",
        "Does the summary avoid inventing facts not found in the article?",
        "Does the summary accurately reflect the author's message?",
        "Does the summary remain concise while covering the key points?"
    ],
)

coherence_metric = GEval(
    name="Coherence",
    model=judge_model,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check whether the summary is logically organized.",
        "Check whether the ideas flow clearly from sentence to sentence.",
        "Check whether the wording is easy to follow for Lawyers.",
        "Check whether the summary avoids repetition.",
        "Check whether the summary reads as a coherent whole."
    ],
)

tonality_metric = GEval(
    name="Tonality",
    model=judge_model,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check whether the tone is clearly Legalese.",
        "Check whether the tone is consistent throughout.",
        "Check whether the word choice matches a legal style.",
        "Check whether the style is distinguishable from plain writing.",
        "Check whether the tone remains readable."
    ],
)

safety_metric = GEval(
    name="Safety",
    model=judge_model,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check whether the summary avoids fabricated claims.",
        "Check whether the summary avoids harmful or offensive language.",
        "Check whether the summary does not misrepresent the article.",
        "Check whether the summary remains grounded in the source text.",
        "Check whether the summary is safe for professional use."
    ],
)


In [8]:
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_results

from IPython.display import display, Markdown

display(Markdown(f"""
## Evaluation Results

**Summarization Score:** {evaluation_results["SummarizationScore"]}  

**Summarization Reason:**  
{evaluation_results["SummarizationReason"]}

**Coherence Score:** {evaluation_results["CoherenceScore"]}  

**Coherence Reason:**  
{evaluation_results["CoherenceReason"]}

**Tonality Score:** {evaluation_results["TonalityScore"]}  

**Tonality Reason:**  
{evaluation_results["TonalityReason"]}

**Safety Score:** {evaluation_results["SafetyScore"]}  

**Safety Reason:**  
{evaluation_results["SafetyReason"]}
"""))

Output()

Output()

Output()

Output()


## Evaluation Results

**Summarization Score:** 0.8461538461538461  

**Summarization Reason:**  
The score is 0.85 because the summary effectively captures the main ideas of the original text, but it introduces extra information about cultivating relationships and adapting to work styles that were not present in the original text.

**Coherence Score:** 0.8049999273946236  

**Coherence Reason:**  
The summary is logically organized and presents a clear flow of ideas, moving from the importance of self-management to specific methodologies and the benefits of relationship cultivation. The wording is generally accessible for lawyers, though some phrases may be slightly complex. There is minimal repetition, and the summary reads as a coherent whole, effectively conveying the main points. However, a few sentences could be simplified for better clarity.

**Tonality Score:** 0.24920184608005652  

**Tonality Reason:**  
The tone of the response does not clearly reflect Legalese; it is more motivational and advisory in nature. While there are some complex phrases, the overall style is not distinctly legal and lacks the formal structure typical of legal documents. Additionally, the readability is compromised by the length and complexity of sentences, making it less accessible than standard legal writing. The word choice does not consistently align with legal terminology, further detracting from its alignment with the evaluation criteria.

**Safety Score:** 0.9143466294357351  

**Safety Reason:**  
The response effectively summarizes the key themes of the article, including the importance of self-management, understanding personal strengths and values, and the necessity of adapting to work environments. It avoids fabricated claims and harmful language, accurately reflects the article's content, and remains grounded in the source text. The emphasis on feedback analysis and relationship management aligns well with the article's core messages, making it suitable for professional use. However, a minor shortcoming is the lack of specific examples from the text that could further enhance the depth of the summary.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [9]:
improved_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": (
                "You are a lawyer revising a summary. Improve the summary using the evaluation feedback. "
                "Keep the tone in Legalese. Keep the summary under 1000 tokens. "
                "Use only information grounded in the source article."
            ),
        },
        {
            "role": "user",
            "content": f"""
ARTICLE:
{document_text}

CURRENT SUMMARY:
{summary_output.Summary}

EVALUATION:
{evaluation_results}
""",
        },
    ],
    text_format=SummaryOutput,
)

improved_summary_output = improved_response.output_parsed
print(improved_summary_output)

from IPython.display import display, Markdown

display(Markdown(f"""
## Structured Summary

**Title:** {improved_summary_output.Title}  
**Author:** {improved_summary_output.Author}  
**Tone:** {improved_summary_output.Tone}  
**Input Tokens:** {improved_summary_output.InputTokens}  
**Output Tokens:** {improved_summary_output.OutputTokens}   

### Relevance
{summary_output.Relevance}

### Summary
{summary_output.Summary}
"""))



Author='Peter F. Drucker' Title='Managing Oneself' Relevance='The article emphasizes the critical need for self-management and self-awareness in the knowledge economy, a vital aspect for AI professionals aiming for sustained career growth.' Summary='In an era defined by unrivaled opportunities, the responsibility for career management has shifted to the individual. Knowledge workers are encouraged to act as their own Chief Executive Officers, necessitating a comprehensive awareness of personal strengths, weaknesses, values, methods of performance, and optimal working environments. The text outlines the feedback analysis method to uncover these insights, asserting that success within the knowledge economy is contingent upon recognizing one’s strengths and aligning with suitable environments. It emphasizes the importance of assessing personal contributions, strategically avoiding assignments that conflict with one’s competencies. The text further advises that effective self-management fa


## Structured Summary

**Title:** Managing Oneself  
**Author:** Peter F. Drucker  
**Tone:** Legalese  
**Input Tokens:** 645  
**Output Tokens:** 780   

### Relevance
This article is significant for AI professionals as it outlines the necessity of self-management and personal accountability in career progression, themes applicable to AI professionals navigating a rapidly evolving and ambiguous job landscape.

### Summary
In the contemporary era characterized by unprecedented opportunities, the onus of career management falls squarely upon the individual. Knowledge workers are urged to adopt the role of their own Chief Executive Officer, thereby necessitating a profound understanding of personal strengths, weaknesses, values, performance modes, and ideal environments for efficacy. The document delineates methodologies such as feedback analysis to uncover these personal insights, emphasizing that successful navigation within the knowledge economy requires acknowledging one’s strengths and actively seeking environments aligned with one’s values and capabilities. Furthermore, individuals are encouraged to cultivate relationships within organizations, enhancing performance through understanding and adapting to the work styles of peers and superiors. The discourse advocates for strategic contributions and ongoing self-development, steering clear of assignments incongruent with personal competencies. This self-aware, proactive approach ensures sustained engagement and productivity throughout an extended career, potentially spanning five decades. Ultimately, the text posits that the ability to manage oneself effectively is not only pivotal for individual excellence but is increasingly necessary in an environment where organizational longevity cannot be assumed.


In [10]:
improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary_output.Summary
)

summarization_metric.measure(improved_test_case)
coherence_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)

improved_evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

improved_evaluation_results

from IPython.display import display, Markdown

display(Markdown(f"""
## Improved Evaluation Results

**Summarization Score:** {improved_evaluation_results["SummarizationScore"]}  

**Summarization Reason:**  
{improved_evaluation_results["SummarizationReason"]}

**Coherence Score:** {improved_evaluation_results["CoherenceScore"]}  

**Coherence Reason:**  
{improved_evaluation_results["CoherenceReason"]}

**Tonality Score:** {improved_evaluation_results["TonalityScore"]}  

**Tonality Reason:**  
{improved_evaluation_results["TonalityReason"]}

**Safety Score:** {improved_evaluation_results["SafetyScore"]}  

**Safety Reason:**  
{improved_evaluation_results["SafetyReason"]}
"""))


Output()

Output()

Output()

Output()


## Improved Evaluation Results

**Summarization Score:** 0.9  

**Summarization Reason:**  
The score is 0.90 because the summary accurately reflects the main points of the original text without contradictions, but it introduces extra information regarding organizational permanence that was not present in the original text.

**Coherence Score:** 0.822270013882531  

**Coherence Reason:**  
The summary is logically organized and presents a clear flow of ideas, moving from the concept of individual responsibility in career management to the importance of self-awareness and effective self-management. The wording is accessible for lawyers, avoiding overly complex language. However, there is slight repetition in emphasizing the need for self-management and awareness, which could be streamlined for better coherence.

**Tonality Score:** 0.24040148391948452  

**Tonality Reason:**  
The tone of the response does not clearly reflect Legalese; it is more motivational and advisory in nature. While it maintains a consistent tone, the word choice lacks the formal and technical language typical of legal writing. The style is not distinguishable from plain writing, as it reads more like a self-help guide than a legal document. Additionally, while the text is readable, it does not meet the criteria for legal tone and style.

**Safety Score:** 0.9143466291110288  

**Safety Reason:**  
The response effectively summarizes the key themes of the article, including the shift in responsibility for career management to individuals and the importance of self-awareness in the knowledge economy. It avoids fabricated claims and harmful language, accurately reflects the article's content, and remains grounded in the source text. The summary is also suitable for professional use, making it a strong alignment with the evaluation steps.


1) The enhanced summary noticably improved summarization, coherence, while keeping safety and tonalty fairly constant.

2) This suggests that pushing the model to improve increased its grounding to the source text and logical flow.

3) These evaluation controls are useful, but they are not enough on their own, as the tonality is not sufficient, making human review still important.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
